# Synchronized montage video exporter (moving-window traces)

Based on `synchronized_video_creation.ipynb`, with the same arena / eye frame synchronization,
but a paper-style moving-window φ / θ / pupil panel (cursor fixed at center).

**Inputs**
- `path_to_block`: path to a `block_NNN` folder
- either `video_start_ms` + `video_end_ms`, or `segments=[(start_ms, end_ms), ...]`
- `half_window_ms`: half-width of the visible trace window around the playhead (default 1500 → ±1.5 s)
- `aspect_ratio`: `"16:9"` or `"4:3"` (grows trace height; videos are not stretched)


In [1]:
from __future__ import annotations

import os
from pathlib import Path
from typing import Dict, List, Literal, Optional, Sequence, Tuple, Union

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib import rcParams
from matplotlib.backends.backend_agg import FigureCanvasAgg
from matplotlib.lines import Line2D
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Rectangle
from matplotlib.ticker import FuncFormatter, MultipleLocator
from tqdm.auto import tqdm

from eye_tracking_system_tools.analysis.eye_trace_io import load_eye_dataframe, resolve_eye_csv
from eye_tracking_system_tools.annotation.block_annotator.block_loader import infer_metadata
from eye_tracking_system_tools.figures.plotting_functions import TRACE_L, TRACE_R
from eye_tracking_system_tools.preprocessing.BlockSync_class import BlockSync
from eye_tracking_system_tools.preprocessing.block_sync_core import load_final_sync_df

%matplotlib inline
plt.style.use("default")
rcParams["pdf.fonttype"] = 42
rcParams["ps.fonttype"] = 42

# Paper-style eye colors.
COLOR_L_HEX = TRACE_L  # left eye blue
COLOR_R_HEX = "#E87722"  # bright orange (right eye)

# Trace panel styling
PLOT_CONTENT_ASPECT_WH = 5.0 / 3.0  # plot white area width:height (w > h)
TRACE_FONT_SCALE = 2.5
TRACE_LINE_WIDTH = 3.5
CURSOR_COLOR = "#E02020"
CURSOR_LINE_WIDTH = 3.0
X_TICK_STEP_S = 1.5  # seconds between scrolling x ticks (~1/1.5 prior density)


## 1. Load a block from `path_to_block`

Point `PATH_TO_BLOCK` at a folder named `block_NNN` that already has `analysis/final_sync_df.csv`
(or `blocksync_df.csv`) and verified eye CSVs.


In [6]:
# --- user inputs ---
#PATH_TO_BLOCK = Path(r"/Volumes/Data/Nimrod/experiments/PV_106/2025_08_06/block_008"), VIDEO_START_MS = 170317,VIDEO_END_MS = 215924  # used before
#PATH_TO_BLOCK = Path(r"/Volumes/Data/Nimrod/experiments/PV_228/2026_06_15/block_012"), VIDEO_START_MS = 47676, VIDEO_END_MS = 58332
PATH_TO_BLOCK = Path(r"/Volumes/Data/Nimrod/experiments/PV_228/2026_05_31/block_002") 
VIDEO_START_MS = 4304 
VIDEO_END_MS = 103387

# Clip timing (ms on the block ms_axis). Use ONE of:
#   A) a single window:

SEGMENTS = None  # e.g. [(240_000, 250_000), (280_000, 290_000)]

#   B) or a list of segments (takes precedence when not None):
# SEGMENTS = [(240_000, 250_000), (280_000, 290_000)]

HALF_WINDOW_MS = 2500  # ±5 s → 10 s window  #ms around the playhead
INTERPOLATE = True  # fill NaN trace gaps with linear-over-time interpolation
PUPIL_ZSCORE = True  # True -> pupil traces are per-eye z-scored (label: pupil [z])
ASPECT_RATIO = "16:9"  # "16:9", "4:3", or None for legacy trace height
OUT_PATH = None  # None → write under block.analysis_path


In [7]:
def recenter_eye_angles_to_rest(
    eye_df: pd.DataFrame,
    phi_col: str = "k_phi",
    theta_col: str = "k_theta",
    inplace: bool = False,
    dropna: bool = True,
) -> Tuple[pd.DataFrame, Dict[str, float]]:
    """Subtract median phi/theta so resting gaze is near zero (same as original exporter)."""
    for c in (phi_col, theta_col):
        if c not in eye_df.columns:
            raise ValueError(f"Column '{c}' not found in dataframe")
    df = eye_df if inplace else eye_df.copy()
    phi_vals = pd.to_numeric(df[phi_col], errors="coerce")
    theta_vals = pd.to_numeric(df[theta_col], errors="coerce")
    phi_med = float(np.nanmedian(phi_vals)) if dropna else float(np.median(phi_vals))
    theta_med = float(np.nanmedian(theta_vals)) if dropna else float(np.median(theta_vals))
    df[phi_col] = phi_vals - phi_med
    df[theta_col] = theta_vals - theta_med
    df[f"{phi_col}_recentered"] = df[phi_col]
    df[f"{theta_col}_recentered"] = df[theta_col]
    return df, {"phi_offset": phi_med, "theta_offset": theta_med}


def prepare_block_for_export(
    path_to_block: Union[os.PathLike, str],
    *,
    eye_csv_left: Optional[Union[os.PathLike, str]] = None,
    eye_csv_right: Optional[Union[os.PathLike, str]] = None,
    calibrate_pixel_size_mm: float = 10.0,
    verbose: bool = True,
) -> BlockSync:
    """Instantiate BlockSync from a block folder and load sync + eye data for export."""
    block_path = Path(path_to_block).resolve()
    if not block_path.is_dir():
        raise FileNotFoundError(f"path_to_block is not a directory: {block_path}")

    animal_call, experiment_date, block_num, path_to_animal = infer_metadata(block_path)
    block = BlockSync(
        animal_call,
        experiment_date,
        block_num,
        str(path_to_animal),
    )
    # Honor the exact folder the user passed (in case layout differs slightly).
    block.block_path = block_path
    block.analysis_path = block_path / "analysis"

    load_final_sync_df(block, verbose=verbose)
    sr = float(getattr(block, "sample_rate", None) or 30000.0)
    if "ms_axis" not in block.final_sync_df.columns:
        block.final_sync_df["ms_axis"] = (
            block.final_sync_df["Arena_TTL"].to_numpy(dtype=float) / (sr / 1000.0)
        )

    if eye_csv_left:
        left_path = Path(eye_csv_left)
        block.left_eye_data = load_eye_dataframe(left_path)
        left_rule = "explicit"
    else:
        left_choice = resolve_eye_csv(block.analysis_path, "left")
        left_path = left_choice.path
        left_rule = left_choice.rule
        block.left_eye_data = load_eye_dataframe(left_path)

    if eye_csv_right:
        right_path = Path(eye_csv_right)
        block.right_eye_data = load_eye_dataframe(right_path)
        right_rule = "explicit"
    else:
        right_choice = resolve_eye_csv(block.analysis_path, "right")
        right_path = right_choice.path
        right_rule = right_choice.rule
        block.right_eye_data = load_eye_dataframe(right_path)

    if verbose:
        print(f"[OK] left eye CSV:  {left_path.name} ({left_rule})")
        print(f"[OK] right eye CSV: {right_path.name} ({right_rule})")

    block.handle_eye_videos()
    block.handle_arena_files()
    try:
        block.calibrate_pixel_size(calibrate_pixel_size_mm)
    except Exception as exc:
        if verbose:
            print(f"[warn] calibrate_pixel_size failed: {exc}")

    if "pupil_diameter" not in block.left_eye_data.columns:
        if verbose:
            print(f"[info] calculating pupil diameter for {block}")
        block.left_eye_data["pupil_diameter_pixels"] = block.left_eye_data.major_ax
        block.right_eye_data["pupil_diameter_pixels"] = block.right_eye_data.major_ax
        l_pix = float(getattr(block, "L_pix_size", 1.0) or 1.0)
        r_pix = float(getattr(block, "R_pix_size", 1.0) or 1.0)
        block.left_eye_data["pupil_diameter"] = block.left_eye_data["pupil_diameter_pixels"] * l_pix
        block.right_eye_data["pupil_diameter"] = block.right_eye_data["pupil_diameter_pixels"] * r_pix

    left_centered, left_off = recenter_eye_angles_to_rest(block.left_eye_data)
    right_centered, right_off = recenter_eye_angles_to_rest(block.right_eye_data)
    block.left_eye_data_centered = left_centered
    block.right_eye_data_centered = right_centered
    if verbose:
        print("Left eye offsets:", left_off)
        print("Right eye offsets:", right_off)
        print(f"[OK] prepared {block}")
    return block


## 2. Export helpers (sync backbone unchanged; visualization updated)

Differences vs the original montage exporter:
1. Paper-style φ / θ / pupil traces (white background, vignette L/R colors, Right/Left legend).
2. Moving time window with a **fixed center cursor** (`half_window_ms`, default ±1.5 s).
3. Accepts a single `[start_ms, end_ms]` or a list of segments concatenated into one MP4.

4. Optional `interpolate=True`: linear-over-time fill for any NaN in φ / θ / pupil traces.
5. `aspect_ratio="16:9"` / `"4:3"` sizes the φ/θ/pupil strip taller so the full frame matches that ratio (no video stretching).
6. `pupil_zscore=True` shows per-eye z-scored pupil diameter (`pupil [z]`) instead of raw mm.
7. Trace plots render in a centered 4:3 white panel (black letterbox); ~3× fonts; red playhead; thicker lines.


In [8]:
class MonotoneFrameReader:
    """
    Frame-exact reader for mostly-nondecreasing frame indices.
    Avoids CAP_PROP_POS_FRAMES random seeking issues with MP4/H264.
    (Unchanged from synchronized_video_creation.ipynb.)
    """

    def __init__(self, path, label="video"):
        self.path = str(path)
        self.label = label
        self.cap = cv2.VideoCapture(self.path)
        if not self.cap.isOpened():
            raise RuntimeError(f"Cannot open {label}: {path}")
        self.cur_idx = -1
        self.cur_frame = None

    def close(self):
        try:
            self.cap.release()
        except Exception:
            pass

    def _reopen_and_seek(self, target_idx: int):
        self.close()
        self.cap = cv2.VideoCapture(self.path)
        if not self.cap.isOpened():
            raise RuntimeError(f"Cannot reopen {self.label}: {self.path}")
        self.cur_idx = -1
        self.cur_frame = None
        if target_idx > 0:
            for _ in range(target_idx):
                ok = self.cap.grab()
                if not ok:
                    return None

    def read_at(self, target_idx: Optional[int]):
        if target_idx is None or target_idx < 0:
            return None
        target_idx = int(target_idx)

        if target_idx == self.cur_idx and self.cur_frame is not None:
            return self.cur_frame

        if target_idx < self.cur_idx:
            self._reopen_and_seek(target_idx)

        while self.cur_idx < target_idx:
            ok, frame = self.cap.read()
            if not ok:
                return None
            self.cur_idx += 1
            self.cur_frame = frame

        return self.cur_frame


_PAPER_SIGNAL_META = (
    ("phi", r"$\phi$ [$^\circ$]", "deg"),
    ("theta", r"$\theta$ [$^\circ$]", "deg"),
    ("pupil_diameter", "pupil [mm]", "pupil"),
)


def _paper_signal_meta(*, pupil_zscore: bool = False) -> Tuple[Tuple[str, str, str], ...]:
    if pupil_zscore:
        return (
            ("phi", r"$\phi$ [$^\circ$]", "deg"),
            ("theta", r"$\theta$ [$^\circ$]", "deg"),
            ("pupil_diameter", "pupil [z]", "pupil_z"),
        )
    return _PAPER_SIGNAL_META


class PaperTraceRenderer:
    """Scrolling trace panel: x-limits move in data space; ticks scroll with traces."""

    def __init__(
        self,
        width_px: int,
        height_px: int,
        half_window_ms: float,
        signal_meta: Sequence[Tuple[str, str, str]] = _PAPER_SIGNAL_META,
        *,
        color_l: str = COLOR_L_HEX,
        color_r: str = COLOR_R_HEX,
        dpi: float = 100.0,
        angle_ylim: Tuple[float, float] = (-15.0, 15.0),
        auto_ylim: bool = False,
        ylim_pad_std: float = 1.0,
        plot_aspect_wh: float = PLOT_CONTENT_ASPECT_WH,
        font_scale: float = TRACE_FONT_SCALE,
        line_width: float = TRACE_LINE_WIDTH,
        cursor_color: str = CURSOR_COLOR,
        cursor_line_width: float = CURSOR_LINE_WIDTH,
        x_tick_step_s: float = X_TICK_STEP_S,
    ):
        self.canvas_width_px = int(width_px)
        self.canvas_height_px = int(height_px)
        self.width_px = self.canvas_width_px
        self.height_px = self.canvas_height_px
        self.plot_height_px = int(height_px)
        self.plot_width_px = int(round(self.plot_height_px * float(plot_aspect_wh)))
        self.half_window_ms = float(half_window_ms)
        self.window_s = 2.0 * self.half_window_ms / 1000.0
        self.x_tick_step_s = float(x_tick_step_s)
        self.signal_meta = tuple(signal_meta)
        self.color_l = color_l
        self.color_r = color_r
        self.dpi = float(dpi)
        self.angle_ylim = angle_ylim
        self.auto_ylim = bool(auto_ylim)
        self.ylim_pad_std = float(ylim_pad_std)
        self._fixed_ylims: Dict[str, Tuple[float, float]] = {}

        self._label_fs = int(round(9 * font_scale))
        self._tick_fs = int(round(8 * font_scale))
        self._legend_fs = int(round(8 * font_scale))
        self._line_width = float(line_width)
        self._cursor_color = cursor_color
        self._cursor_line_width = float(cursor_line_width)
        spine_lw = max(1.0, 0.8 * font_scale / 3.0)
        self._spine_lw = spine_lw

        self._x_locator = MultipleLocator(self.x_tick_step_s)
        self._x_formatter = FuncFormatter(lambda v, _pos: f"{v:.1f}")

        w_in = self.canvas_width_px / self.dpi
        h_in = self.canvas_height_px / self.dpi
        self.fig = plt.figure(figsize=(w_in, h_in), dpi=self.dpi, facecolor="black")
        FigureCanvasAgg(self.fig)

        plot_w_frac = min(1.0, self.plot_width_px / max(1, self.canvas_width_px))
        legend_w_frac = min(0.14, max(0.08, 120.0 / max(1, self.canvas_width_px)))
        content_w_frac = min(0.98, plot_w_frac + legend_w_frac)
        content_x0 = (1.0 - content_w_frac) / 2.0
        self._plot_x0_frac = content_x0
        self._plot_w_frac = plot_w_frac

        panel_bottom = 0.02
        panel_top = 0.98
        self._white_panel = Rectangle(
            (content_x0, panel_bottom),
            content_w_frac,
            panel_top - panel_bottom,
            transform=self.fig.transFigure,
            facecolor="white",
            edgecolor="none",
            zorder=0,
            clip_on=False,
        )
        self.fig.add_artist(self._white_panel)

        plot_x0 = content_x0
        # Extra left inset keeps y-labels away from the black letterbox.
        inner_left = plot_x0 + 0.16 * plot_w_frac
        inner_right = plot_x0 + 0.94 * plot_w_frac
        inner_bottom = 0.16
        inner_top = 0.94
        hspace = 0.07 if len(self.signal_meta) > 1 else 0.0

        gs = GridSpec(
            nrows=len(self.signal_meta),
            ncols=1,
            figure=self.fig,
            left=inner_left,
            right=inner_right,
            top=inner_top,
            bottom=inner_bottom,
            hspace=hspace,
        )

        self._lines_l: List = []
        self._lines_r: List = []
        self.axes = []
        legend_handles = None
        legend_labels = None

        for i, (name, ylabel, _kind) in enumerate(self.signal_meta):
            ax = self.fig.add_subplot(gs[i, 0])
            ax.set_facecolor("white")
            ax.patch.set_alpha(1.0)
            ax.set_zorder(2)
            ax.set_autoscale_on(False)
            (ln_r,) = ax.plot(
                [], [], color=self.color_r, lw=self._line_width,
                label="Right", solid_capstyle="round",
            )
            (ln_l,) = ax.plot(
                [], [], color=self.color_l, lw=self._line_width,
                label="Left", solid_capstyle="round",
            )
            ax.set_ylabel(ylabel, fontsize=self._label_fs)
            ax.yaxis.set_label_coords(-0.12, 0.5)
            ax.tick_params(
                axis="y", direction="out", labelsize=self._tick_fs,
                width=spine_lw * 0.8, length=4 * font_scale / 3.0,
            )
            for spine in ("top", "right"):
                ax.spines[spine].set_visible(False)
            ax.spines["left"].set_linewidth(spine_lw)
            ax.spines["bottom"].set_linewidth(spine_lw)

            if i == len(self.signal_meta) - 1:
                ax.set_xlabel("Time [s]", fontsize=self._label_fs)
                ax.tick_params(axis="x", labelsize=self._tick_fs, pad=6)
                ax.xaxis.set_major_locator(self._x_locator)
                ax.xaxis.set_major_formatter(self._x_formatter)
            else:
                ax.tick_params(axis="x", labelbottom=False)

            if i == 0:
                legend_handles, legend_labels = ax.get_legend_handles_labels()

            self._lines_r.append(ln_r)
            self._lines_l.append(ln_l)
            self.axes.append(ax)

        self.fig.align_ylabels(self.axes)

        legend_x = plot_x0 + plot_w_frac + 0.008
        self._legend = self.fig.legend(
            legend_handles,
            legend_labels,
            loc="upper left",
            bbox_to_anchor=(legend_x, inner_top),
            frameon=True,
            facecolor="white",
            edgecolor="none",
            framealpha=1.0,
            fontsize=self._legend_fs,
            handlelength=1.8,
            borderaxespad=0.4,
        )
        for txt in self._legend.get_texts():
            txt.set_color("black")

        self._cursor_line = Line2D(
            [0.0, 0.0], [0.0, 0.0],
            transform=self.fig.transFigure,
            color=self._cursor_color,
            lw=self._cursor_line_width,
            zorder=200,
            clip_on=False,
        )
        self.fig.add_artist(self._cursor_line)

        self._set_xlim(0.0, self.window_s)
        for ax in self.axes:
            ax.set_ylim(-1.0, 1.0)
        self.fig.canvas.draw()
        self._fit_white_panel_to_content()
        self.fig.canvas.draw()
        self._cache_cursor_y_span()

    def close(self):
        plt.close(self.fig)

    def _fit_white_panel_to_content(self, left_pad: float = 0.03, pad: float = 0.012) -> None:
        """Expand white backdrop once; extra left pad clears black letterbox from y-labels."""
        renderer = self.fig.canvas.get_renderer()
        bboxes = [
            ax.get_tightbbox(renderer).transformed(self.fig.transFigure.inverted())
            for ax in self.axes
        ]
        if getattr(self, "_legend", None) is not None:
            bboxes.append(
                self._legend.get_window_extent(renderer).transformed(self.fig.transFigure.inverted())
            )
        if not bboxes:
            return
        x0 = min(b.x0 for b in bboxes) - left_pad
        x1 = max(b.x1 for b in bboxes) + pad
        y0 = min(b.y0 for b in bboxes) - pad
        y1 = max(b.y1 for b in bboxes) + pad
        x0 = max(0.0, x0)
        y0 = max(0.0, y0)
        x1 = min(1.0, x1)
        y1 = min(1.0, y1)
        self._white_panel.set_x(x0)
        self._white_panel.set_y(y0)
        self._white_panel.set_width(max(0.01, x1 - x0))
        self._white_panel.set_height(max(0.01, y1 - y0))

    def _cache_cursor_y_span(self) -> None:
        pos_top = self.axes[0].get_position()
        pos_bot = self.axes[-1].get_position()
        self._cursor_y_bot = float(pos_bot.y0)
        self._cursor_y_top = float(pos_top.y1)

    def _set_xlim(self, x0: float, x1: float) -> None:
        for ax in self.axes:
            ax.set_xlim(x0, x1)

    def _update_cursor(self, t_cur_s: float) -> None:
        ax_ref = self.axes[-1]
        x_disp = ax_ref.transData.transform((t_cur_s, 0.0))
        x_fig = float(self.fig.transFigure.inverted().transform(x_disp)[0])
        self._cursor_line.set_data([x_fig, x_fig], [self._cursor_y_bot, self._cursor_y_top])

    @staticmethod
    def _range_with_margin(vals: np.ndarray, fallback: Tuple[float, float], margin_frac: float = 0.05):
        x = np.asarray(vals, dtype=float)
        x = x[np.isfinite(x)]
        if x.size == 0:
            return fallback
        lo = float(np.min(x))
        hi = float(np.max(x))
        span = hi - lo
        pad = margin_frac * span if span > 1e-12 else max(abs(lo), abs(hi), 1.0) * margin_frac
        lo -= pad
        hi += pad
        if abs(hi - lo) < 1e-9:
            lo, hi = lo - 1.0, hi + 1.0
        return lo, hi

    def set_fixed_ylims(self, ylim_by_signal: Dict[str, Tuple[float, float]]) -> None:
        self._fixed_ylims = dict(ylim_by_signal)

    @staticmethod
    def compute_segment_ylims(
        t_grid_ms: np.ndarray,
        Lsig: Dict[str, np.ndarray],
        Rsig: Dict[str, np.ndarray],
        seg_start_ms: float,
        seg_end_ms: float,
        signal_meta: Sequence[Tuple[str, str, str]],
        *,
        margin_frac: float = 0.05,
        angle_fallback: Tuple[float, float] = (-15.0, 15.0),
        pupil_fallback: Tuple[float, float] = (0.0, 1.0),
        pupil_z_fallback: Tuple[float, float] = (-3.5, 3.5),
    ) -> Dict[str, Tuple[float, float]]:
        mask = (t_grid_ms >= seg_start_ms) & (t_grid_ms <= seg_end_ms)
        out: Dict[str, Tuple[float, float]] = {}
        for name, _ylabel, kind in signal_meta:
            Lv = Lsig.get(name, np.array([]))[mask] if name in Lsig else np.array([])
            Rv = Rsig.get(name, np.array([]))[mask] if name in Rsig else np.array([])
            vals = np.concatenate([np.asarray(Lv, float), np.asarray(Rv, float)])
            if kind == "deg":
                fb = angle_fallback
            elif kind == "pupil_z":
                fb = pupil_z_fallback
            else:
                fb = pupil_fallback
            out[name] = PaperTraceRenderer._range_with_margin(vals, fb, margin_frac=margin_frac)
        return out

    def render(
        self,
        t_ms: float,
        t_grid_ms: np.ndarray,
        Lsig: Dict[str, np.ndarray],
        Rsig: Dict[str, np.ndarray],
    ) -> np.ndarray:
        half = self.half_window_ms
        w0, w1 = float(t_ms) - half, float(t_ms) + half
        mask = (t_grid_ms >= w0) & (t_grid_ms <= w1)
        tg = t_grid_ms[mask]
        t_s = tg / 1000.0
        t_cur_s = float(t_ms) / 1000.0
        x0, x1 = w0 / 1000.0, w1 / 1000.0

        self._set_xlim(x0, x1)

        for i, (name, _ylabel, kind) in enumerate(self.signal_meta):
            ax = self.axes[i]
            Lv = Lsig[name][mask] if name in Lsig else np.full(tg.shape, np.nan)
            Rv = Rsig[name][mask] if name in Rsig else np.full(tg.shape, np.nan)
            self._lines_l[i].set_data(t_s, Lv)
            self._lines_r[i].set_data(t_s, Rv)

            if name in self._fixed_ylims:
                ax.set_ylim(*self._fixed_ylims[name])
            elif kind == "deg":
                ax.set_ylim(*self.angle_ylim)
            elif kind == "pupil_z":
                ax.set_ylim(-3.0, 3.0)
            else:
                ax.set_ylim(0.0, 1.0)

        self._update_cursor(t_cur_s)

        self.fig.canvas.draw()
        buf = np.asarray(self.fig.canvas.buffer_rgba())
        rgb = buf[:, :, :3]
        if rgb.shape[0] != self.canvas_height_px or rgb.shape[1] != self.canvas_width_px:
            rgb = cv2.resize(
                rgb, (self.canvas_width_px, self.canvas_height_px), interpolation=cv2.INTER_AREA,
            )
        return cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)



def _zscore_array(y: np.ndarray) -> np.ndarray:
    """Per-series z-score; mean/std from finite values only."""
    y = np.asarray(y, dtype=float)
    finite = np.isfinite(y)
    if finite.sum() < 2:
        return y.copy()
    mu = float(np.mean(y[finite]))
    sd = float(np.std(y[finite], ddof=0))
    if sd < 1e-12:
        out = np.zeros_like(y, dtype=float)
        out[~finite] = np.nan
        return out
    out = (y - mu) / sd
    out[~finite] = np.nan
    return out


def _apply_pupil_zscore(
    Lsig: Dict[str, np.ndarray],
    Rsig: Dict[str, np.ndarray],
    *,
    signal: str = "pupil_diameter",
) -> Tuple[Dict[str, np.ndarray], Dict[str, np.ndarray]]:
    Lout = dict(Lsig)
    Rout = dict(Rsig)
    if signal in Lout:
        Lout[signal] = _zscore_array(Lout[signal])
    if signal in Rout:
        Rout[signal] = _zscore_array(Rout[signal])
    return Lout, Rout


def _interpolate_signal_over_time(t_ms: np.ndarray, y: np.ndarray) -> np.ndarray:
    """Linear interpolation over time (ms); interior gaps only, edges stay NaN."""
    t = np.asarray(t_ms, dtype=float)
    y = np.asarray(y, dtype=float).copy()
    finite = np.isfinite(y) & np.isfinite(t)
    if finite.sum() < 2:
        return y
    bad = ~finite
    if not bad.any():
        return y
    y[bad] = np.interp(t[bad], t[finite], y[finite], left=np.nan, right=np.nan)
    return y


def _interpolate_trace_dict_over_time(
    t_ms: np.ndarray,
    Lsig: Dict[str, np.ndarray],
    Rsig: Dict[str, np.ndarray],
    signals: Sequence[str],
) -> Tuple[Dict[str, np.ndarray], Dict[str, np.ndarray]]:
    """Fill NaN gaps in each trace using linear interpolation over ms_axis."""
    Lout = dict(Lsig)
    Rout = dict(Rsig)
    for sig in signals:
        if sig in Lout:
            Lout[sig] = _interpolate_signal_over_time(t_ms, Lout[sig])
        if sig in Rout:
            Rout[sig] = _interpolate_signal_over_time(t_ms, Rout[sig])
    return Lout, Rout


_ASPECT_RATIO_PRESETS = {
    "16:9": 16.0 / 9.0,
    "4:3": 4.0 / 3.0,
}


def _resolve_aspect_ratio(aspect_ratio: Optional[Union[str, float]]) -> Optional[float]:
    """Return width/height for a named preset or numeric ratio; None keeps legacy layout."""
    if aspect_ratio is None:
        return None
    if isinstance(aspect_ratio, (int, float)):
        r = float(aspect_ratio)
        if r <= 0:
            raise ValueError(f"aspect_ratio must be > 0, got {aspect_ratio}")
        return r
    key = str(aspect_ratio).strip().lower().replace(" ", "")
    if key not in _ASPECT_RATIO_PRESETS:
        raise ValueError(
            f"Unknown aspect_ratio={aspect_ratio!r}. Use one of "
            f"{sorted(_ASPECT_RATIO_PRESETS)} or a numeric width/height."
        )
    return _ASPECT_RATIO_PRESETS[key]


def _trace_height_for_aspect(
    width_px: int,
    banner_h: int,
    video_row_h: int,
    aspect_wh: float,
    *,
    min_trace_h: int,
) -> Tuple[int, int, float]:
    """
    Choose trace panel height so total frame matches width/height = aspect_wh.

    Videos are never stretched: only the trace strip grows (or is floored at
    ``min_trace_h`` if the video+banner already exceed the target height).
    Returns (trace_h, Htotal, achieved_wh).
    """
    target_h = int(round(float(width_px) / float(aspect_wh)))
    fixed = int(banner_h) + int(video_row_h)
    needed = target_h - fixed
    if needed < int(min_trace_h):
        trace_h = int(min_trace_h)
        Htotal = fixed + trace_h
    else:
        trace_h = int(needed)
        Htotal = target_h
    achieved = float(width_px) / float(Htotal) if Htotal > 0 else float("nan")
    return trace_h, Htotal, achieved


def _normalize_segments(
    start_ms: Optional[float],
    end_ms: Optional[float],
    segments: Optional[Sequence[Tuple[float, float]]],
) -> List[Tuple[float, float]]:
    if segments is not None:
        out: List[Tuple[float, float]] = []
        for i, seg in enumerate(segments):
            if len(seg) != 2:
                raise ValueError(f"segments[{i}] must be (start_ms, end_ms), got {seg!r}")
            s, e = float(seg[0]), float(seg[1])
            if e <= s:
                raise ValueError(f"segments[{i}] end_ms must be > start_ms (got {s}, {e})")
            out.append((s, e))
        if not out:
            raise ValueError("segments is empty")
        return out
    if start_ms is None or end_ms is None:
        raise ValueError("Provide either segments=[...], or both start_ms and end_ms")
    s, e = float(start_ms), float(end_ms)
    if e <= s:
        raise ValueError(f"end_ms must be > start_ms (got start_ms={s}, end_ms={e})")
    return [(s, e)]


def export_block_synchronized_montage_video_moving_window(
    block: object,
    out_path: Union[os.PathLike, str],
    *,
    start_ms: Optional[float] = None,
    end_ms: Optional[float] = None,
    segments: Optional[Sequence[Tuple[float, float]]] = None,
    fps: float = 60.0,
    half_window_ms: float = 1500.0,
    arena_video: Optional[Union[int, str]] = None,
    arena_frame_cols: Sequence[str] = (
        "Arena_frame", "arena_frame", "arena_frames", "arena_frame_idx",
        "frame", "frame_idx", "video_frame", "arena_idx",
    ),
    arena_frame_shift: int = 0,
    L_eye_frame_cols: Sequence[str] = (
        "L_eye_frame", "left_eye_frame", "le_eye_frame", "L_frame", "le_frame", "L_eye_idx",
    ),
    R_eye_frame_cols: Sequence[str] = (
        "R_eye_frame", "right_eye_frame", "re_eye_frame", "R_frame", "re_frame", "R_eye_idx",
    ),
    eye_video_mode: Literal["auto", "raw", "dlc"] = "raw",
    dlc_name_hint: str = "DLC",
    top_banner_h: int = 60,
    banner_title: str = "Synchronized Video",
    trace_h: int = 280,
    trace_scale: float = 1.0,
    aspect_ratio: Optional[Union[str, float]] = "16:9",
    min_trace_h: Optional[int] = None,
    flip_eyes_vertical: bool = True,
    trace_signals: Sequence[str] = ("phi", "theta", "pupil_diameter"),
    trace_col_map: Optional[Dict[str, str]] = None,
    use_centered_eye_data: bool = True,
    disqualify_cols: Sequence[str] = ("center_x", "center_y"),
    show_disqualified_badge: bool = True,
    require_all_three: bool = True,
    codec: str = "mp4v",
    timestamp_precision_ms: int = 0,
    show_debug_prints: bool = True,
    angle_ylim: Tuple[float, float] = (-15.0, 15.0),
    ylim_margin_frac: float = 0.05,
    interpolate: bool = False,
    pupil_zscore: bool = False,
    auto_ylim: bool = False,
) -> Path:
    """
    Export a synchronized Right | Arena | Left montage with a paper-style
    moving-window phi/theta/pupil panel (playhead fixed at the horizontal center).

    Timing
    ------
    - Single clip: pass ``start_ms`` and ``end_ms``.
    - Multi-clip: pass ``segments=[(start_ms, end_ms), ...]``; segments are
      written back-to-back into one MP4.

    Visualization
    -------------
    - ``half_window_ms`` controls the visible half-width around the playhead
      (default 1500 -> always show +/- 1.5 s of high-resolution traces).
    - Y limits are computed once per segment from the full segment range (+ ``ylim_margin_frac``).
    - When ``interpolate=True``, NaN values in each trace column are filled with
      linear-over-time interpolation on the sync ms axis (interior gaps only).
    - When ``pupil_zscore=True``, pupil diameter is z-scored per eye over the
      export trace context (after optional interpolation).
    - ``aspect_ratio`` (``"16:9"``, ``"4:3"``, or numeric W/H) grows the trace
      panel height so the full frame matches that ratio without stretching videos.
      Pass ``None`` to keep the legacy ``trace_h * trace_scale`` layout.
    """

    def _require_attr(obj: object, name: str):
        if not hasattr(obj, name):
            raise AttributeError(f"block is missing required attribute '{name}'")
        return getattr(obj, name)

    def _get_attr_optional(obj: object, name: str, default=None):
        return getattr(obj, name, default)

    def _require_cols(df: pd.DataFrame, cols: Sequence[str], df_name: str):
        missing = [c for c in cols if c not in df.columns]
        if missing:
            raise ValueError(f"{df_name} is missing required columns: {missing}")

    def _pick_first_video(paths, name: str) -> Path:
        if paths is None:
            raise ValueError(f"block.{name} is None")
        if isinstance(paths, (str, os.PathLike)):
            p = Path(paths)
            if not p.exists():
                raise FileNotFoundError(p)
            return p
        if isinstance(paths, (list, tuple)) and len(paths) > 0:
            p = Path(paths[0])
            if not p.exists():
                raise FileNotFoundError(p)
            return p
        raise ValueError(f"block.{name} has no usable video path(s)")

    def _open_cap(p: Path, label: str) -> cv2.VideoCapture:
        cap = cv2.VideoCapture(str(p))
        if not cap.isOpened():
            raise RuntimeError(f"Cannot open {label} video: {p}")
        return cap

    def _resolve_col(df: pd.DataFrame, candidates: Sequence[str], what: str) -> str:
        for c in candidates:
            if c in df.columns:
                return c
        raise ValueError(
            f"final_sync_df has no recognizable {what} column. Tried: {list(candidates)}"
        )

    def _safe_put_text(img, text, org, color, scale=0.6, thickness=2):
        x, y = org
        cv2.putText(
            img, text, (x + 1, y + 1), cv2.FONT_HERSHEY_SIMPLEX, scale, (0, 0, 0),
            thickness + 2, cv2.LINE_AA,
        )
        cv2.putText(
            img, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX, scale, color, thickness, cv2.LINE_AA,
        )

    def _make_banner(W: int, title: str) -> np.ndarray:
        banner = np.zeros((top_banner_h, W, 3), dtype=np.uint8)
        (tw, _), _ = cv2.getTextSize(title, cv2.FONT_HERSHEY_SIMPLEX, 0.9, 2)
        x = max(12, (W - tw) // 2)
        _safe_put_text(banner, title, (x, 34), (255, 255, 255), scale=0.9, thickness=2)
        return banner

    def _clamp_idx(idx: Optional[int], cap: cv2.VideoCapture) -> Optional[int]:
        if idx is None:
            return None
        n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
        if n > 0:
            return int(np.clip(int(idx), 0, n - 1))
        return max(0, int(idx))

    def _resize_to_height(img: np.ndarray, target_h: int) -> np.ndarray:
        h, w = img.shape[:2]
        if h == target_h:
            return img
        new_w = max(1, int(round(w * (target_h / float(h)))))
        return cv2.resize(img, (new_w, target_h), interpolation=cv2.INTER_AREA)

    def _choose_arena_video(arena_list: Sequence[str], choice: Optional[Union[int, str]]) -> Path:
        paths = [Path(p) for p in arena_list]
        if not paths:
            raise ValueError("block.arena_videos is empty.")
        if isinstance(choice, int):
            if choice < 0 or choice >= len(paths):
                raise ValueError(f"arena_video index {choice} out of range (0..{len(paths)-1})")
            return paths[int(choice)]
        if isinstance(choice, str) and choice.strip():
            key = choice.strip().lower()
            hits = [p for p in paths if key in p.name.lower()]
            if not hits:
                raise ValueError(f"arena_video='{choice}' did not match any arena video name.")
            hits.sort(key=lambda p: len(p.name))
            return hits[0]
        if len(paths) == 1:
            return paths[0]
        import sys
        if not sys.stdin.isatty():
            return paths[0]
        print("\nSelect arena video:")
        for i, p in enumerate(paths):
            print(f"  [{i}] {p.name}")
        raw = ""
        try:
            raw = input("Enter arena index (default 0): ").strip()
        except Exception:
            raw = ""
        idx = 0 if raw == "" else int(raw)
        if idx < 0 or idx >= len(paths):
            raise ValueError(f"arena index {idx} out of range.")
        return paths[idx]

    def _resolve_eye_video(raw_path: Path, mode: str) -> Path:
        raw_path = Path(raw_path)
        if mode == "raw":
            return raw_path
        folder = raw_path.parent
        if not folder.exists():
            if mode == "dlc":
                raise FileNotFoundError(f"Eye video folder not found: {folder}")
            return raw_path
        hint = dlc_name_hint.lower()
        dlc_candidates = [p for p in folder.glob("*.mp4") if hint in p.name.lower()]
        stem = raw_path.stem.lower()
        stem_hits = [p for p in dlc_candidates if stem in p.stem.lower()]
        candidates = stem_hits if stem_hits else dlc_candidates
        if candidates:
            candidates.sort(key=lambda p: p.name)
            return candidates[0]
        if mode == "dlc":
            raise FileNotFoundError(
                f"eye_video_mode='dlc' but no DLC mp4 found in {folder} (hint='{dlc_name_hint}')"
            )
        return raw_path

    segs = _normalize_segments(start_ms, end_ms, segments)
    half_window_ms = float(half_window_ms)
    if half_window_ms <= 0:
        raise ValueError("half_window_ms must be > 0")

    fsync = _require_attr(block, "final_sync_df")
    if not isinstance(fsync, pd.DataFrame):
        raise ValueError("block.final_sync_df is not a pandas DataFrame.")

    arena_fcol = _resolve_col(fsync, arena_frame_cols, "arena frame")
    L_fcol = _resolve_col(fsync, L_eye_frame_cols, "LEFT eye frame")
    R_fcol = _resolve_col(fsync, R_eye_frame_cols, "RIGHT eye frame")

    if "ms_axis" in fsync.columns:
        ms_all = fsync["ms_axis"].to_numpy(dtype=float)
    elif "Arena_TTL" in fsync.columns:
        sr = float(_get_attr_optional(block, "sample_rate", None) or block.get_sample_rate())
        ms_all = fsync["Arena_TTL"].to_numpy(dtype=float) / sr * 1000.0
    elif "oe_time_s" in fsync.columns:
        ms_all = fsync["oe_time_s"].to_numpy(dtype=float) * 1000.0
    else:
        raise ValueError(
            "final_sync_df has no 'ms_axis' and no usable time column ('Arena_TTL' or 'oe_time_s')."
        )

    left_df_raw: pd.DataFrame = _require_attr(block, "left_eye_data")
    right_df_raw: pd.DataFrame = _require_attr(block, "right_eye_data")
    _require_cols(left_df_raw, ["eye_frame"], "left_eye_data")
    _require_cols(right_df_raw, ["eye_frame"], "right_eye_data")

    left_centered = _get_attr_optional(block, "left_eye_data_centered", None)
    right_centered = _get_attr_optional(block, "right_eye_data_centered", None)
    if use_centered_eye_data:
        if left_centered is None or right_centered is None:
            raise AttributeError(
                "use_centered_eye_data=True but block.left_eye_data_centered / "
                "block.right_eye_data_centered not found. Run prepare_block_for_export() first."
            )
        left_df, right_df = left_centered, right_centered
        _require_cols(left_df, ["eye_frame"], "left_eye_data_centered")
        _require_cols(right_df, ["eye_frame"], "right_eye_data_centered")
    else:
        left_df, right_df = left_df_raw, right_df_raw

    if use_centered_eye_data:
        default_map = {
            "pupil_diameter": "pupil_diameter",
            "phi": "k_phi_recentered",
            "theta": "k_theta_recentered",
        }
    else:
        default_map = {
            "pupil_diameter": "pupil_diameter",
            "phi": "k_phi",
            "theta": "k_theta",
        }
    if trace_col_map is None:
        col_map = default_map
    else:
        col_map = default_map.copy()
        col_map.update(trace_col_map)

    paper_order = ["phi", "theta", "pupil_diameter"]
    trace_signals = [s for s in paper_order if s in set(trace_signals)] or list(trace_signals)
    signal_meta = tuple(
        meta for meta in _paper_signal_meta(pupil_zscore=pupil_zscore)
        if meta[0] in set(trace_signals)
    )

    Ltab = left_df.drop_duplicates(subset=["eye_frame"], keep="first").set_index("eye_frame", drop=False)
    Rtab = right_df.drop_duplicates(subset=["eye_frame"], keep="first").set_index("eye_frame", drop=False)

    def _signals_for_frames(L_frames: np.ndarray, R_frames: np.ndarray, t_ms: np.ndarray):
        L_eye_idx = pd.Index(L_frames, name="eye_frame")
        R_eye_idx = pd.Index(R_frames, name="eye_frame")
        Lsig: Dict[str, np.ndarray] = {}
        Rsig: Dict[str, np.ndarray] = {}
        for sig in trace_signals:
            col = col_map.get(sig, sig)
            if (col not in Ltab.columns) or (col not in Rtab.columns):
                Lsig[sig] = np.full_like(t_ms, np.nan, dtype=float)
                Rsig[sig] = np.full_like(t_ms, np.nan, dtype=float)
                continue
            Lsig[sig] = Ltab.reindex(L_eye_idx)[col].to_numpy(dtype=float)
            Rsig[sig] = Rtab.reindex(R_eye_idx)[col].to_numpy(dtype=float)
        return Lsig, Rsig

    def _disq_flags(tab: pd.DataFrame, eye_idx: pd.Index) -> np.ndarray:
        if not disqualify_cols:
            return np.zeros(len(eye_idx), dtype=bool)
        sub = tab.reindex(eye_idx)
        flags = np.zeros(len(eye_idx), dtype=bool)
        for c in disqualify_cols:
            if c in sub.columns:
                flags |= sub[c].isna().to_numpy()
        return flags

    # Trace context includes +/- half_window around all segments so the cursor stays centered.
    global_t0 = min(s for s, _ in segs) - half_window_ms
    global_t1 = max(e for _, e in segs) + half_window_ms
    ctx_mask = np.isfinite(ms_all) & (ms_all >= global_t0) & (ms_all <= global_t1)
    if not np.any(ctx_mask):
        raise ValueError("No final_sync_df rows fall inside the requested time range (+/- half window).")
    ms_ctx = ms_all[ctx_mask].astype(float)
    fs_ctx = fsync.iloc[np.where(ctx_mask)[0]].copy()
    L_ctx = fs_ctx[L_fcol].to_numpy(dtype=float)
    R_ctx = fs_ctx[R_fcol].to_numpy(dtype=float)
    L_frames_ctx = np.array([int(v) if np.isfinite(v) else -1 for v in L_ctx], dtype=np.int64)
    R_frames_ctx = np.array([int(v) if np.isfinite(v) else -1 for v in R_ctx], dtype=np.int64)
    Lsig_ctx, Rsig_ctx = _signals_for_frames(L_frames_ctx, R_frames_ctx, ms_ctx)
    if interpolate:
        Lsig_ctx, Rsig_ctx = _interpolate_trace_dict_over_time(
            ms_ctx, Lsig_ctx, Rsig_ctx, trace_signals,
        )
        if show_debug_prints:
            print("[traces] applied linear-over-time interpolation on sync ms axis")
    if pupil_zscore and "pupil_diameter" in trace_signals:
        Lsig_ctx, Rsig_ctx = _apply_pupil_zscore(Lsig_ctx, Rsig_ctx)
        if show_debug_prints:
            print("[traces] pupil diameter z-scored per eye over export context")

    rv_raw = _pick_first_video(_require_attr(block, "re_videos"), "re_videos")
    lv_raw = _pick_first_video(_require_attr(block, "le_videos"), "le_videos")
    rv = _resolve_eye_video(rv_raw, eye_video_mode)
    lv = _resolve_eye_video(lv_raw, eye_video_mode)
    arena_list = _require_attr(block, "arena_videos")
    arena_path = _choose_arena_video(arena_list, arena_video)

    capR = _open_cap(rv, "right_eye")
    capL = _open_cap(lv, "left_eye")
    capA = _open_cap(arena_path, "arena")
    rR = MonotoneFrameReader(rv, "right_eye")
    rL = MonotoneFrameReader(lv, "left_eye")
    rA = MonotoneFrameReader(arena_path, "arena")

    Wr, Hr = int(capR.get(cv2.CAP_PROP_FRAME_WIDTH)), int(capR.get(cv2.CAP_PROP_FRAME_HEIGHT))
    Wl, Hl = int(capL.get(cv2.CAP_PROP_FRAME_WIDTH)), int(capL.get(cv2.CAP_PROP_FRAME_HEIGHT))
    Wa, Ha = int(capA.get(cv2.CAP_PROP_FRAME_WIDTH)), int(capA.get(cv2.CAP_PROP_FRAME_HEIGHT))

    Heye = int(min(Hr, Hl))
    Wr_out = max(1, int(round(Wr * (Heye / float(Hr)))))
    Wl_out = max(1, int(round(Wl * (Heye / float(Hl)))))
    Wa_out = max(1, int(round(Wa * (Heye / float(Ha)))))
    Hrow = Heye
    Wtotal = Wr_out + Wa_out + Wl_out

    legacy_trace_h = max(1, int(round(float(trace_h) * float(trace_scale))))
    min_trace = int(min_trace_h) if min_trace_h is not None else legacy_trace_h
    aspect_wh = _resolve_aspect_ratio(aspect_ratio)
    if aspect_wh is None:
        trace_h_eff = legacy_trace_h
        Htotal = top_banner_h + Hrow + trace_h_eff
        achieved_wh = float(Wtotal) / float(Htotal)
        if show_debug_prints:
            print(
                f"[layout] legacy traces: W={Wtotal} H={Htotal} "
                f"(~{achieved_wh:.3f}:1, neither forced)"
            )
    else:
        trace_h_eff, Htotal, achieved_wh = _trace_height_for_aspect(
            Wtotal,
            top_banner_h,
            Hrow,
            aspect_wh,
            min_trace_h=min_trace,
        )
        if show_debug_prints:
            print(
                f"[layout] target aspect={aspect_ratio} ({aspect_wh:.4f}:1) | "
                f"output={Wtotal}x{Htotal} (~{achieved_wh:.4f}:1) | "
                f"trace_h={trace_h_eff} (video_row={Hrow}, banner={top_banner_h})"
            )
            if abs(achieved_wh - aspect_wh) > 0.01:
                print(
                    "[layout][warn] could not hit target aspect without shrinking videos; "
                    f"used min_trace_h={min_trace}. "
                    "Pick a taller target (e.g. 4:3) or reduce eye video height."
                )

    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    writer = cv2.VideoWriter(
        str(out_path),
        cv2.VideoWriter_fourcc(*codec),
        float(fps),
        (Wtotal, Htotal),
    )
    if not writer.isOpened():
        for cap in (capR, capL, capA):
            cap.release()
        raise RuntimeError(f"Could not open VideoWriter for: {out_path} (codec='{codec}')")

    banner = _make_banner(Wtotal, banner_title)
    renderer = PaperTraceRenderer(
        width_px=Wtotal,
        height_px=trace_h_eff,
        half_window_ms=half_window_ms,
        signal_meta=signal_meta,
        color_l=COLOR_L_HEX,
        color_r=COLOR_R_HEX,
        angle_ylim=angle_ylim,
        auto_ylim=auto_ylim,
    )

    if show_debug_prints:
        print(f"[export] segments={segs}")
        print(f"[export] half_window_ms={half_window_ms} | output={Wtotal}x{Htotal} | fps={fps}")
        print(f"[export] arena={arena_path.name} | RE={Path(rv).name} | LE={Path(lv).name}")
        src = "centered" if use_centered_eye_data else "raw"
        pupil_mode = "z-score" if pupil_zscore else "raw mm"
        print(
            f"[traces] source={src} | signals={list(trace_signals)} | cols={col_map} | "
            f"pupil={pupil_mode}"
        )

    prev_R = np.zeros((Hr, Wr, 3), dtype=np.uint8)
    prev_L = np.zeros((Hl, Wl, 3), dtype=np.uint8)
    prev_A = np.zeros((Ha, Wa, 3), dtype=np.uint8)

    def _subsample_segment(seg_start: float, seg_end: float):
        mask = np.isfinite(ms_all) & (ms_all >= seg_start) & (ms_all <= seg_end)
        if not np.any(mask):
            raise ValueError(f"No final_sync_df rows in segment [{seg_start}, {seg_end}].")
        idx_rows = np.where(mask)[0]
        fs_win = fsync.iloc[idx_rows].copy()
        t_ms = ms_all[idx_rows].astype(float)

        if len(t_ms) > 5:
            dt = np.median(np.diff(t_ms))
            fps_master = 1000.0 / dt if dt > 0 else float("nan")
            if np.isfinite(fps_master) and fps_master > 0:
                stride = int(round(fps_master / float(fps))) if float(fps) <= fps_master else 1
                stride = max(1, stride)
                if stride > 1:
                    fs_win = fs_win.iloc[::stride].copy()
                    t_ms = t_ms[::stride]
                    if show_debug_prints:
                        print(
                            f"[export] segment [{seg_start:.0f},{seg_end:.0f}] "
                            f"master fps≈{fps_master:.2f}; stride={stride}"
                        )

        A_raw = fs_win[arena_fcol].to_numpy(dtype=float)
        L_raw = fs_win[L_fcol].to_numpy(dtype=float)
        R_raw = fs_win[R_fcol].to_numpy(dtype=float)
        if require_all_three:
            ok = np.isfinite(A_raw) & np.isfinite(L_raw) & np.isfinite(R_raw)
            fs_win = fs_win.loc[ok].copy()
            t_ms = t_ms[ok]
            A_raw, L_raw, R_raw = A_raw[ok], L_raw[ok], R_raw[ok]
            if len(fs_win) < 5:
                raise RuntimeError(
                    f"Too few valid rows in segment [{seg_start}, {seg_end}] "
                    "after require_all_three filtering."
                )
        if arena_frame_shift != 0:
            A_raw = A_raw + float(int(arena_frame_shift))

        A_frames = np.array([int(v) if np.isfinite(v) else -1 for v in A_raw], dtype=np.int64)
        L_frames = np.array([int(v) if np.isfinite(v) else -1 for v in L_raw], dtype=np.int64)
        R_frames = np.array([int(v) if np.isfinite(v) else -1 for v in R_raw], dtype=np.int64)
        disqL = _disq_flags(Ltab, pd.Index(L_frames, name="eye_frame"))
        disqR = _disq_flags(Rtab, pd.Index(R_frames, name="eye_frame"))
        return t_ms, A_frames, L_frames, R_frames, disqL, disqR

    try:
        total_frames = 0
        for seg_i, (seg_start, seg_end) in enumerate(segs):
            seg_ylims = PaperTraceRenderer.compute_segment_ylims(
                ms_ctx,
                Lsig_ctx,
                Rsig_ctx,
                seg_start,
                seg_end,
                signal_meta,
                margin_frac=ylim_margin_frac,
                angle_fallback=angle_ylim,
            )
            renderer.set_fixed_ylims(seg_ylims)
            if show_debug_prints:
                print(f"[ylims] segment {seg_i + 1}: {seg_ylims}")

            t_ms, A_frames, L_frames, R_frames, disqL, disqR = _subsample_segment(seg_start, seg_end)
            desc = f"Segment {seg_i + 1}/{len(segs)}"
            for i in tqdm(range(len(t_ms)), desc=desc, unit="frame", dynamic_ncols=True):
                tcur = float(t_ms[i])

                idxR = int(R_frames[i]) if R_frames[i] >= 0 else None
                idxL = int(L_frames[i]) if L_frames[i] >= 0 else None
                idxA = int(A_frames[i]) if A_frames[i] >= 0 else None
                idxR = _clamp_idx(idxR, capR)
                idxL = _clamp_idx(idxL, capL)
                idxA = _clamp_idx(idxA, capA)

                missingR = missingL = missingA = False
                fR = rR.read_at(idxR) if idxR is not None else None
                if fR is None:
                    fR = prev_R.copy()
                    missingR = True
                else:
                    prev_R = fR.copy()
                fL = rL.read_at(idxL) if idxL is not None else None
                if fL is None:
                    fL = prev_L.copy()
                    missingL = True
                else:
                    prev_L = fL.copy()
                fA = rA.read_at(idxA) if idxA is not None else None
                if fA is None:
                    fA = prev_A.copy()
                    missingA = True
                else:
                    prev_A = fA.copy()

                if flip_eyes_vertical:
                    fR = cv2.flip(fR, 0)
                    fL = cv2.flip(fL, 0)

                _safe_put_text(fR, "RIGHT", (12, 24), (255, 255, 255), scale=0.75, thickness=2)
                _safe_put_text(fA, "ARENA", (12, 24), (255, 255, 255), scale=0.75, thickness=2)
                _safe_put_text(fL, "LEFT", (12, 24), (255, 255, 255), scale=0.75, thickness=2)

                if show_disqualified_badge and disqR[i]:
                    _safe_put_text(
                        fR, "disqualified", (max(10, fR.shape[1] - 170), 24),
                        (0, 0, 255), scale=0.65, thickness=2,
                    )
                if show_disqualified_badge and disqL[i]:
                    _safe_put_text(
                        fL, "disqualified", (max(10, fL.shape[1] - 170), 24),
                        (0, 0, 255), scale=0.65, thickness=2,
                    )
                if missingR:
                    _safe_put_text(fR, "missing frame", (12, fR.shape[0] - 12), (255, 0, 0), scale=0.6, thickness=2)
                if missingL:
                    _safe_put_text(fL, "missing frame", (12, fL.shape[0] - 12), (255, 0, 0), scale=0.6, thickness=2)
                if missingA:
                    _safe_put_text(fA, "missing frame", (12, fA.shape[0] - 12), (255, 0, 0), scale=0.6, thickness=2)

                ts_str = f"{tcur:.{timestamp_precision_ms}f} ms"
                ts_sz, _ = cv2.getTextSize(ts_str, cv2.FONT_HERSHEY_SIMPLEX, 0.85, 2)
                tx = max(12, (fA.shape[1] - ts_sz[0]) // 2)
                ty = max(28, fA.shape[0] - 12)
                _safe_put_text(fA, ts_str, (tx, ty), (255, 255, 255), scale=0.85, thickness=2)

                fR = _resize_to_height(fR, Heye)
                fA = _resize_to_height(fA, Heye)
                fL = _resize_to_height(fL, Heye)
                row_img = np.concatenate([fR, fA, fL], axis=1)
                if row_img.shape[1] != Wtotal:
                    if row_img.shape[1] < Wtotal:
                        row_img = cv2.copyMakeBorder(
                            row_img, 0, 0, 0, Wtotal - row_img.shape[1],
                            cv2.BORDER_CONSTANT, value=(0, 0, 0),
                        )
                    else:
                        row_img = row_img[:, :Wtotal, :]

                trace = renderer.render(tcur, ms_ctx, Lsig_ctx, Rsig_ctx)

                frame = np.zeros((Htotal, Wtotal, 3), dtype=np.uint8)
                frame[0:top_banner_h, :, :] = banner
                frame[top_banner_h:top_banner_h + Hrow, :, :] = row_img
                frame[top_banner_h + Hrow:top_banner_h + Hrow + trace_h_eff, :, :] = trace
                writer.write(frame)
                total_frames += 1

        if show_debug_prints:
            print(f"[OK] wrote {total_frames} frames -> {out_path}")
        return out_path
    finally:
        renderer.close()
        rR.close()
        rL.close()
        rA.close()
        try:
            writer.release()
        except Exception:
            pass
        for cap in (capR, capL, capA):
            try:
                cap.release()
            except Exception:
                pass


## 3. Run export

Edit paths / times in the config cell above, then run this cell.


In [9]:
block = prepare_block_for_export(PATH_TO_BLOCK)

if OUT_PATH is None:
    if SEGMENTS is not None:
        tag = f"segs{len(SEGMENTS)}"
    else:
        tag = f"{int(VIDEO_START_MS)}_{int(VIDEO_END_MS)}"
    OUT_PATH = Path(block.analysis_path) / f"montage_moving_window_{tag}.mp4"

out = export_block_synchronized_montage_video_moving_window(
    block,
    out_path=OUT_PATH,
    start_ms=None if SEGMENTS is not None else VIDEO_START_MS,
    end_ms=None if SEGMENTS is not None else VIDEO_END_MS,
    segments=SEGMENTS,
    half_window_ms=HALF_WINDOW_MS,
    fps=60.0,
    eye_video_mode="raw",
    arena_frame_shift=0,  
    require_all_three=True,
    interpolate=INTERPOLATE,
    pupil_zscore=PUPIL_ZSCORE,
    aspect_ratio=ASPECT_RATIO,
    banner_title="Synchronized Video",
    arena_video=2
)
print("Saved:", out)


instantiated block number 002 at Path: /Volumes/Data/Nimrod/experiments/PV_228/2026_05_31/block_002, new OE version
Found the sample rate for block 002 in the xml file, it is 20000 Hz

Extracting meta data from: /Volumes/Data/Nimrod/experiments/PV_228/2026_05_31/block_002/oe_files/PV228_d1t2_hunter_22026-05-31_15-25-27/Record Node 106...
Analog channel numbers contain duplicates!!! Reordering numbers serially.

Extracting time stamp information...

Error!!! Some blocks are missing in recording!!!

Checking integrity of all records in ch1...

Metadata extraction complete.
created the .oe_rec attribute as an open ephys recording obj with get_data functionality (standalone mode, no metadata file)
retrieving zertoh sample number for block 002
got it!
[INFO] Found multiple sync files: ['final_sync_df.csv', 'blocksync_df.csv']. Using newest: final_sync_df.csv
[OK] Loaded final_sync_df.csv -> block.final_sync_df (rows=36,580)
[OK] left eye CSV:  left_eye_data_raw_verified.csv (raw_verified)
[

Segment 1/1:   0%|          | 0/4338 [00:00<?, ?frame/s]

[OK] wrote 4338 frames -> /Volumes/Data/Nimrod/experiments/PV_228/2026_05_31/block_002/analysis/montage_moving_window_4304_103387.mp4
Saved: /Volumes/Data/Nimrod/experiments/PV_228/2026_05_31/block_002/analysis/montage_moving_window_4304_103387.mp4


### Multi-segment example

```python
out = export_block_synchronized_montage_video_moving_window(
    block,
    out_path=Path(block.analysis_path) / "montage_segments.mp4",
    segments=[(240_000, 250_000), (280_000, 290_000), (310_000, 320_000)],
    half_window_ms=1500.0,  # ±1.5 s
    aspect_ratio="4:3",  # or "16:9"
    fps=60.0,
)
```
